# Module 00: Environment Verification Lab

This notebook verifies that your ChipWhisperer development environment is correctly configured. Run each cell sequentially and confirm all checks pass before proceeding to Module 01.

**Note:** Hardware-specific checks will produce warnings if no ChipWhisperer device is connected — this is expected for simulated labs.

In [ ]:
import sys
import platform
import subprocess

print("=" * 60)
print("ENVIRONMENT VERIFICATION REPORT")
print("=" * 60)

# 1. Python version
py_version = sys.version_info
status = "PASS" if py_version >= (3, 10) else "FAIL"
print(f"[{status}] Python {py_version.major}.{py_version.minor}.{py_version.micro}")
print(f"       Platform: {platform.platform()}")
print(f"       Architecture: {platform.machine()}")

# 2. Virtual environment check
import os
venv_active = hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
status = "PASS" if venv_active else "WARN"
print(f"[{status}] Virtual environment active: {venv_active}")
print(f"       Prefix: {sys.prefix}")

# 3. NumPy
try:
    import numpy as np
    print(f"[PASS] NumPy {np.__version__}")
except ImportError:
    print("[FAIL] NumPy not installed")

# 4. Matplotlib
try:
    import matplotlib
    print(f"[PASS] Matplotlib {matplotlib.__version__}")
except ImportError:
    print("[FAIL] Matplotlib not installed")

# 5. Jupyter
try:
    import jupyter
    print(f"[PASS] Jupyter available")
except ImportError:
    print("[WARN] Jupyter not importable (may still work as CLI)")

In [ ]:
# Verify ChipWhisperer installation
print("=" * 60)
print("CHIPWHISPERER VERIFICATION")
print("=" * 60)

try:
    import chipwhisperer as cw
    print(f"[PASS] chipwhisperer version: {cw.__version__}")
    print(f"       Module location: {cw.__file__}")
except ImportError as e:
    print(f"[FAIL] Cannot import chipwhisperer: {e}")
    print("       Run: uv pip install chipwhisperer")

# Check submodules
submodules = [
    'chipwhisperer.analyzer',
    'chipwhisperer.capture',
    'chipwhisperer.common',
]
for mod_name in submodules:
    try:
        __import__(mod_name)
        print(f"[PASS] {mod_name} importable")
    except ImportError as e:
        print(f"[WARN] {mod_name} not importable: {e}")

In [ ]:
# Hardware detection (optional — warning if no device)
print("=" * 60)
print("HARDWARE DETECTION")
print("=" * 60)

try:
    import chipwhisperer as cw
    scope = cw.scope()
    print(f"[PASS] Scope connected: {scope}")
    print(f"       Firmware version: {scope.fw_version}")
    scope.dis()
except Exception as e:
    print(f"[INFO] No hardware detected (expected in simulation mode)")
    print(f"       Error: {e}")

# Check libusb availability
import ctypes
import ctypes.util

libusb_path = ctypes.util.find_library('usb-1.0')
if libusb_path:
    print(f"[PASS] libusb found: {libusb_path}")
else:
    libusb_path = ctypes.util.find_library('usb')
    if libusb_path:
        print(f"[PASS] libusb found: {libusb_path}")
    else:
        print("[WARN] libusb not found via ctypes (may still work)")
        print("       Install: brew install libusb (macOS) / apt install libusb-1.0-0-dev (Linux)")

In [ ]:
# Cross-compiler verification
print("=" * 60)
print("CROSS-COMPILER VERIFICATION")
print("=" * 60)

compilers = {
    'arm-none-eabi-gcc': 'ARM Embedded (STM32, etc.)',
    'avr-gcc': 'AVR (ATmega328P)',
}

for compiler, desc in compilers.items():
    try:
        result = subprocess.run(
            [compiler, '--version'],
            capture_output=True, text=True, timeout=5
        )
        version_line = result.stdout.split('\n')[0]
        print(f"[PASS] {compiler} ({desc}): {version_line}")
    except FileNotFoundError:
        print(f"[WARN] {compiler} ({desc}) not found")
        print(f"       Install: apt install gcc-arm-none-eabi / brew install avr-gcc")
    except subprocess.TimeoutExpired:
        print(f"[WARN] {compiler} timed out")

print("\n" + "=" * 60)
print("ENVIRONMENT VERIFICATION COMPLETE")
print("=" * 60)
print("If all checks show [PASS], you are ready for Module 01.")
print("[WARN] items are non-fatal and acceptable for simulated labs.")